In [1]:
import warnings
warnings.filterwarnings("ignore")
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import pickle
import wandb
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

MODEL_NAME = 'ViT'   # <-- change per notebook
GPU_ID = 0               # <-- change to 1 for a second notebook running in parallel

os.environ['CUDA_VISIBLE_DEVICES'] = str(GPU_ID)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device} (physical GPU {GPU_ID})")

with open('../data/data_config.pkl', 'rb') as f:
    config = pickle.load(f)

train_df      = config['train_df']
val_df        = config['val_df']
test_df       = config['test_df']
CLASSES       = config['classes']
class_weights = config['class_weights']
IMAGE_SIZE    = config['image_size']
BATCH_SIZE    = config['batch_size']

SEG_DIR = '../data/segmented_images'


assert os.path.isdir(SEG_DIR)
n_seg = len(os.listdir(SEG_DIR))
print(f"Segmented images found: {n_seg:,}")

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}  Classes: {len(CLASSES)}")
print(f"CLASSES = {CLASSES}")

Device : cuda (physical GPU 0)
Segmented images found: 112,120
Train: 80,726  Val: 8,970  Test: 22,424  Classes: 15
CLASSES = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia', 'No Finding']


In [2]:
wandb.init(
    project='xray-classification',
    name=f'03-classification-{MODEL_NAME}',
    config={
        'model': MODEL_NAME,
        'image_size': IMAGE_SIZE,
        'batch_size': BATCH_SIZE,
        'seg_source': SEG_DIR,
        'num_classes': len(CLASSES)
    }
)
print(f"✅ Wandb run started: 03-classification-{MODEL_NAME}")

wandb: Currently logged in as: chandinigunna06 (chandinigunna06-iiest-shibpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ Wandb run started: 03-classification-ViT


In [3]:
FILENAME_COL = 'Image Index'
LABEL_COL    = 'Finding Labels'

def build_multihot(df, classes):
    label_lists = df[LABEL_COL].str.split('|')
    multihot = np.zeros((len(df), len(classes)), dtype=np.float32)
    class_to_idx = {c: i for i, c in enumerate(classes)}
    for row_idx, labels in enumerate(label_lists):
        for lbl in labels:
            lbl = lbl.strip()
            if lbl in class_to_idx:
                multihot[row_idx, class_to_idx[lbl]] = 1.0
    return multihot

class NIHClassificationDataset(Dataset):
    def __init__(self, df, image_dir, classes, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.classes = classes
        self.transform = transform
        self.labels = build_multihot(self.df, classes)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fname = self.df.iloc[idx][FILENAME_COL]
        img = Image.open(os.path.join(self.image_dir, fname)).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx])

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = NIHClassificationDataset(train_df, SEG_DIR, CLASSES, train_transform)
val_dataset   = NIHClassificationDataset(val_df,   SEG_DIR, CLASSES, val_transform)
test_dataset  = NIHClassificationDataset(test_df,  SEG_DIR, CLASSES, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=8, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")
print(f"Label matrix shape: {train_dataset.labels.shape}")
print("Positive counts per class (train):")
for c, cnt in zip(CLASSES, train_dataset.labels.sum(axis=0)):
    print(f"  {c:20s}: {int(cnt):,}")

Train batches: 2523 | Val: 281 | Test: 701
Label matrix shape: (80726, 15)
Positive counts per class (train):
  Atelectasis         : 8,388
  Cardiomegaly        : 2,012
  Effusion            : 9,673
  Infiltration        : 14,388
  Mass                : 4,168
  Nodule              : 4,551
  Pneumonia           : 960
  Pneumothorax        : 3,828
  Consolidation       : 3,395
  Edema               : 1,673
  Emphysema           : 1,817
  Fibrosis            : 1,243
  Pleural_Thickening  : 2,455
  Hernia              : 160
  No Finding          : 43,329


In [4]:
from torch.utils.data import WeightedRandomSampler

# Vectorized: weight vector aligned to CLASSES order
weight_vec = np.array([class_weights[c] for c in CLASSES])   # shape (15,)

# Per-sample weight = max weight among the classes present (labels is 0/1 multi-hot)
sample_weights = (train_dataset.labels * weight_vec).max(axis=1)

print(f"Sample weight range: {sample_weights.min():.3f} to {sample_weights.max():.3f}")
print(f"Mean sample weight : {sample_weights.mean():.3f}")

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# Rebuild train_loader with sampler instead of shuffle
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,          # <-- replaces shuffle=True
    num_workers=8,
    pin_memory=True
)

print(f"✅ Weighted sampler active — train_loader now oversamples rare classes")
print(f"   (val_loader / test_loader stay as plain sequential loaders — no sampler on eval)")

Sample weight range: 0.157 to 42.517
Mean sample weight : 1.014
✅ Weighted sampler active — train_loader now oversamples rare classes
   (val_loader / test_loader stay as plain sequential loaders — no sampler on eval)


In [5]:
import os
os.environ['HF_TOKEN'] = 'REDACTED_HF_TOKEN' 

In [6]:
import timm
import torch.nn as nn

model = timm.create_model(
    'vit_base_patch16_224',
    pretrained=True,
    num_classes=len(CLASSES)
)

model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"✅ ViT-Base loaded → {n_params:,} parameters")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=1e-5
)
criterion = nn.BCEWithLogitsLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2
)

✅ ViT-Base loaded → 85,810,191 parameters


In [7]:
import time

CHECKPOINT_DIR = f'../checkpoints/{MODEL_NAME}'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'latest.pt')
BEST_PATH = os.path.join(CHECKPOINT_DIR, 'best.pt')

NUM_EPOCHS = 50
EARLY_STOP_PATIENCE = 7     # stop if val AUC doesn't improve for 7 straight epochs
start_epoch = 0
best_val_auc = 0.0
epochs_since_improvement = 0

if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch = ckpt['epoch'] + 1
    best_val_auc = ckpt['best_val_auc']
    epochs_since_improvement = ckpt.get('epochs_since_improvement', 0)
    print(f"🔄 Resumed from epoch {start_epoch} (best val AUC so far: {best_val_auc:.4f})")
else:
    print("🆕 Starting fresh training run")


def run_epoch(loader, training=True):
    model.train() if training else model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, labels in tqdm(loader, desc="Train" if training else "Val"):
            images, labels = images.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            all_preds.append(torch.sigmoid(outputs).detach().cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    aucs = [roc_auc_score(all_labels[:, i], all_preds[:, i])
            for i in range(len(CLASSES)) if all_labels[:, i].sum() > 0]
    return total_loss / len(loader), np.mean(aucs)


for epoch in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()
    train_loss, train_auc = run_epoch(train_loader, training=True)
    val_loss, val_auc = run_epoch(val_loader, training=False)
    scheduler.step(val_auc)
    elapsed = time.time() - t0

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} AUC: {train_auc:.4f} | "
          f"Val Loss: {val_loss:.4f} AUC: {val_auc:.4f} | "
          f"{elapsed:.0f}s")

    wandb.log({
        'epoch': epoch + 1, 'train_loss': train_loss, 'train_auc': train_auc,
        'val_loss': val_loss, 'val_auc': val_auc, 'lr': optimizer.param_groups[0]['lr']
    })

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        epochs_since_improvement = 0
        torch.save({'epoch': epoch, 'model_state': model.state_dict(), 'val_auc': val_auc}, BEST_PATH)
        print(f"  💾 New best model saved (val AUC: {val_auc:.4f})")
    else:
        epochs_since_improvement += 1

    torch.save({
        'epoch': epoch, 'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(), 'scheduler_state': scheduler.state_dict(),
        'best_val_auc': best_val_auc, 'epochs_since_improvement': epochs_since_improvement
    }, CHECKPOINT_PATH)

    if epochs_since_improvement >= EARLY_STOP_PATIENCE:
        print(f"\n⏹️ Early stopping — no improvement for {EARLY_STOP_PATIENCE} epochs")
        break

wandb.log({'final_best_val_auc': best_val_auc})
print(f"\n✅ Training complete! Best val AUC: {best_val_auc:.4f}")

🆕 Starting fresh training run


Val: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 281/281 [00:30<00:00,  9.13it/s]


Epoch 1/50 | Train Loss: 0.2954 AUC: 0.7617 | Val Loss: 0.2494 AUC: 0.7584 | 898s
  💾 New best model saved (val AUC: 0.7584)


Val: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 281/281 [00:31<00:00,  8.85it/s]


Epoch 2/50 | Train Loss: 0.2442 AUC: 0.8519 | Val Loss: 0.2360 AUC: 0.7573 | 929s


...

Epoch 6/50 | Train Loss: 0.0753 AUC: 0.9829 | Val Loss: 0.2847 AUC: 0.7113 | 900s


Val: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 281/281 [00:30<00:00,  9.07it/s]


Epoch 7/50 | Train Loss: 0.0615 AUC: 0.9877 | Val Loss: 0.2905 AUC: 0.7059 | 898s


Val: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 281/281 [00:31<00:00,  8.79it/s]


Epoch 8/50 | Train Loss: 0.0432 AUC: 0.9927 | Val Loss: 0.3131 AUC: 0.7080 | 949s

⏹️ Early stopping — no improvement for 7 epochs

✅ Training complete! Best val AUC: 0.7584


In [1]:
# Load the best checkpoint (epoch 1's weights, which had the best val AUC)
best_ckpt = torch.load(BEST_PATH, map_location=device, weights_only=False)
model.load_state_dict(best_ckpt['model_state'])
model.eval()

print(f"Loaded best checkpoint from epoch {best_ckpt['epoch']+1} (val AUC: {best_ckpt['val_auc']:.4f})")

test_loss, test_auc = run_epoch(test_loader, training=False)

# Per-class test AUC breakdown
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Final test eval"):
        images = images.to(device)
        outputs = model(images)
        all_preds.append(torch.sigmoid(outputs).cpu().numpy())
        all_labels.append(labels.numpy())
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

per_class_auc = {}
for i, c in enumerate(CLASSES):
    if all_labels[:, i].sum() > 0:
        per_class_auc[c] = roc_auc_score(all_labels[:, i], all_preds[:, i])
    else:
        per_class_auc[c] = None

print(f"\n{'='*50}")
print(f"TEST RESULTS — {MODEL_NAME}")
print(f"{'='*50}")
print(f"Overall Test Loss: {test_loss:.4f}")
print(f"Overall Test AUC : {test_auc:.4f}")
print(f"\nPer-class Test AUC:")
for c, auc in sorted(per_class_auc.items(), key=lambda x: -(x[1] or 0)):
    print(f"  {c:20s}: {auc:.4f}" if auc is not None else f"  {c:20s}: N/A")

wandb.log({
    'test_loss': test_loss,
    'test_auc': test_auc,
    **{f'test_auc_{c}': v for c, v in per_class_auc.items() if v is not None}
})

# Save results into a shared pickle for the comparison notebook
import pickle
RESULTS_PATH = '../data/class_results.pkl'

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, 'rb') as f:
        all_results = pickle.load(f)
else:
    all_results = {}

all_results[MODEL_NAME] = {
    'val_auc': best_ckpt['val_auc'],
    'test_auc': test_auc,
    'test_loss': test_loss,
    'per_class_auc': per_class_auc,
    'best_epoch': best_ckpt['epoch'] + 1
}

with open(RESULTS_PATH, 'wb') as f:
    pickle.dump(all_results, f)

print(f"\n✅ Results saved to {RESULTS_PATH}")
print(f"Models recorded so far: {list(all_results.keys())}")

wandb.finish()

NameError: name 'torch' is not defined